<a href="https://colab.research.google.com/github/SaipoojithaNamani/AnimateItNow/blob/main/Training_(Gen_AI)_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**RAG**

In [1]:
!pip install wikipedia faiss-cpu sentence-transformers transformers torch numpy

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 125.7 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=32dfc3d8ea4d75f5e4a185af339f32a1d121881d54cc9587ee6cd1bc18fbb804
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


Step 2: Fetch and Chunk Data
We use the Wikipedia-API to get content and then split it into small "chunks." This is crucial because models have a limited "context window" and cannot read an entire Wikipedia page at once

In [5]:
import wikipedia

# Set language for wikipedia library
wikipedia.set_lang("en")

def get_wiki_content(page_title):
    try:
        # Use wikipedia.page directly. The 'wikipedia' library handles language globally.
        # auto_suggest=False and redirect=True are good practices for direct page fetching.
        page = wikipedia.page(page_title, auto_suggest=False, redirect=True)
        return page.content # Full content is in .content, not .text for the 'wikipedia' library
    except wikipedia.exceptions.PageError:
        print(f"Page '{page_title}' does not exist.")
        return ""
    except wikipedia.exceptions.DisambiguationError as e:
        print(f"Disambiguation error for '{page_title}'. Options: {e.options}")
        # For simplicity, we'll try to retrieve content for the first suggestion.
        try:
            page = wikipedia.page(e.options[0], auto_suggest=False, redirect=True)
            return page.content
        except wikipedia.exceptions.PageError:
            print(f"Could not retrieve content for '{e.options[0]}' from disambiguation options.")
            return ""
        except Exception as inner_e:
            print(f"An unexpected error occurred during disambiguation handling: {inner_e}")
            return ""
    except Exception as e:
        print(f"An unexpected error occurred while fetching '{page_title}': {e}")
        return ""

# Strategy: Split text into 500-character chunks with some overlap
def create_chunks(text, chunk_size=500, overlap=100):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i : i + chunk_size])
    return chunks

full_text = get_wiki_content("Artificial intelligence")
chunks = create_chunks(full_text)
print(f"Created {len(chunks)} chunks.")

Created 224 chunks.


In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load T5 Model
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Load SentenceTransformer for embeddings
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for the chunks
# Ensure 'chunks' variable is available from previous cells
chunk_embeddings = embedding_model.encode(chunks)

# Create a FAISS index
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # L2 distance for similarity
index.add(chunk_embeddings) # Add the embeddings to the index

def retrieve_context(query, k=3):
    # Embed the query
    query_embedding = embedding_model.encode([query])

    # Search the FAISS index for the top k most similar chunks
    distances, indices = index.search(query_embedding, k)

    # Retrieve the actual text chunks
    retrieved_chunks = [chunks[i] for i in indices[0]]

    # Combine chunks into a single context string
    context = "\n\n".join(retrieved_chunks)
    return context

def ask_rag(query):
    context = retrieve_context(query)
    # The 'Prompt' that forces the model to use the context
    prompt = f"Using the context below, answer the question.\n\nContext: {context}\n\nQuestion: {query}"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test it
print(ask_rag("Who coined the term Artificial Intelligence?"))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

a paper 'Computing Machinery and Intelligence'
